In [ ]:
#import package and set up api key
import os
import sys
sys.path.append("..")
from util import check_api_key
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import json

openai_api_key = check_api_key("openai")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
#DB-backed price lookups now come from AirlineDbUtil's shared module instead of local definitions
from airline_db_util import DB, get_ticket_price, set_ticket_price

In [19]:
#test data inserted from dictionary

get_ticket_price("London")
get_ticket_price("SYDNEY")
get_ticket_price("mumbai")

DB TOOL CALLED: Getting price for London
Ticket price to London is $799.0
DB TOOL CALLED: Getting price for SYDNEY
Ticket price to SYDNEY is $2999.0
DB TOOL CALLED: Getting price for mumbai
No price data available for this city


In [21]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

save_price_function = {
    "name": "set_ticket_price",
        "description": "Set the price of a return ticket to the destination city.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to",
                },
                "price" : {
                    "type": "integer",
                    "description": "Ticket price to the city that the customer wants to travel to",
                }
            },
            "required": ["destination_city", "price"],
            "additionalProperties": False
        }
}

tools = [{"type" : "function", "function" : price_function},{"type" : "function", "function" : save_price_function}]


In [37]:
#handle tool 

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        print(f"Handle tool call being called for {tool_call.function.name}")
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role" : "tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            print(f"Calling set_ticket_price with city={city} and price={price}")
            price_details = set_ticket_price(city, price)
            responses.append({
                "role" : "tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
    return responses

In [43]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model = MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Tools calls is going to be executed for messages: {message}")
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return response.choices[0].message.content

In [39]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


Tools calls is going to be executed for messages: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_KrJUQz5L84D3hmRuC1PVBv0d', function=Function(arguments='{"destination_city": "London"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_cfgXTGVZn0tzQ2OP8TTckpR4', function=Function(arguments='{"destination_city": "Paris"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_tns2eXB3Sj0igbRkuZDroEb3', function=Function(arguments='{"destination_city": "Berlin"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_qY0E9RPJPYtg1iuNkqCQRWvq', function=Function(arguments='{"destination_city": "Rome"}', name='get_ticket_price'), type='function')])
Handle tool call being called for get_ticket_price
DB TOOL CALLED: Getting price for London
Handle tool 